#### 그래프 실행
- RunnableConfig
    - recursion_limit: 최대 노드 실행 개수 지정 (순환 로직에 빠지지 않기 위함)
    - thread_id: 그래프 실행 아이디 기록, 추후 추적하기 위한 목적으로 활용됨 (multi-turn 대화할 때, 이 id 별로 따로 대화내용 저장)
    
- 상태(State)로 시작
    - 여기서 "question"에 질문만 입력하고 상태를 첫 번째 노드에 전달
- invoke(상태, config) 전달하여 실행

In [1]:
from graph import build_graph
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage

config = RunnableConfig(
    configurable={
        "thread_id": "research-001",
    }
)

inputs = {"messages": [HumanMessage(content="physics informed neural network for fault detection in airplane")]}
graph = build_graph()
output = graph.invoke(inputs, config)

print(output["messages"][-1].content)

/scratch/hpc201a03/.conda/envs/gapag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangSmith 추적을 시작합니다.
[프로젝트명]
GAPAGO
ROUTE: NEED_USER_CLARIFICATION

query_proposal: "physics-informed neural networks for aircraft fault detection and diagnosis using sensor data"

missing_slots:
- aircraft subsystem scope (e.g., flight control, propulsion, structural health, avionics)
- fault type (e.g., actuator faults, sensor faults, structural damage, system-level anomalies)
- data modality (e.g., simulated data, flight test data, real-time onboard sensors)
- PINN formulation focus (e.g., governing equations, constraints, hybrid models)
- time horizon (e.g., recent 5 years vs. historical)
- application stage (fault detection vs. isolation vs. prognosis)

clarify_questions:
- Which aircraft subsystem or component are you interested in analyzing for faults?
- Are you focusing on sensor faults, actuator faults, structural damage, or general anomaly detection?
- Should the studies use real flight data, simulations, or both?
- Do you want emphasis on methodological development of PINNs 

In [2]:
# 그래프 상태 스냅샷 생성
snapshot = graph.get_state(config)

# 다음 스냅샷 상태 접근
snapshot.next

('human_clarify',)

In [3]:
graph.update_state(
    config,
    {
        "query_approved": True,
        "ask_human": False,
        "messages": [HumanMessage(content="APPROVE")],
    },
)

for event in graph.stream(None, config, stream_mode="values"):
    if "messages" in event:
        print("==>", event["messages"][-1].content)

==> APPROVE
==> ```json
{
  "expanded_terms": [
    "physics-informed neural network",
    "PINN",
    "physics-informed machine learning",
    "hybrid physics–data model",
    "aircraft fault detection",
    "airplane fault diagnosis",
    "aerospace fault detection",
    "aircraft system health monitoring",
    "aircraft diagnostics",
    "prognostics and health management (PHM)",
    "electric aircraft propulsion diagnostics"
  ],
  "papers": [
    {
      "paper_id": "NASA-TechPort-146394",
      "title": "Physics-Informed Neural Network for Next-Gen Electric Aircraft",
      "year": 2022,
      "url": "https://techport.nasa.gov/projects/146394",
      "abstract": "Development of a physics-informed neural network surrogate model for next-generation electric aircraft propulsion system diagnostics, fault detection, and remaining useful life prediction, embedding partially known physics into neural network blocks.",
      "score_bm25": 0.0,
      "source": "NASA TechPort"
    }
  ],
 

In [ ]:
output2 = graph.invoke(None, config)
print(output2["messages"][-1].content)